# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Chạy trên **Kaggle (GPU T4, Internet On)**. Hàm chính nằm trong `lab_lib.py` (cùng thư mục). Đọc `README.md` và `GUIDE.md` trước.

## 0. Môi trường (Kaggle)
Add Input: dataset chứa code lab **và** dataset `road-damage-dataset-potholes-cracks-and-manholes`.

In [ ]:
# Chỉ dùng 1 GPU (T4 x2 -> dùng GPU 0). PHẢI chạy trước khi import torch.
import os, shutil, pathlib
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
if not pathlib.Path("lab_lib.py").exists():   # chép code lab từ /kaggle/input sang nơi ghi được
    src = next(pathlib.Path("/kaggle/input").rglob("lab_lib.py")).parent
    shutil.copytree(src, "/kaggle/working/lab", dirs_exist_ok=True)
    os.chdir("/kaggle/working/lab")
print(os.getcwd())

In [ ]:
!nvidia-smi -L
!pip -q install -r requirements-colab.txt
# Sau khi cài xong: Run > Restart session, rồi chạy lại cell trên, bỏ qua cell pip này.

## 1. Dữ liệu: tải và chia split

In [ ]:
import pathlib
SRC = next(pathlib.Path("/kaggle/input").rglob("labels-YOLO")).parent   # thư mục có images/ và labels-YOLO/
print(SRC)
!python scripts/prepare_data.py --source {SRC}
!python -m pytest -q

In [ ]:
import json, random
from PIL import Image
import matplotlib.pyplot as plt, matplotlib.patches as pa
from lab_lib import *

# Không session nào được nằm ở hai split
sess = {s: {e["session"] for e in read_jsonl(f"data/splits/{s}.jsonl")} for s in ("train", "val", "test")}
print("train∩val:", sess["train"] & sess["val"], "| train∩test:", sess["train"] & sess["test"], "| val∩test:", sess["val"] & sess["test"])

# Vẽ 6 ảnh có box (đi qua JSON 0-1000 rồi quy về pixel -> kiểm tra luôn bước 2)
rows = read_jsonl("data/splits/train.jsonl")
fig, axs = plt.subplots(2, 3, figsize=(15, 6))
for ax, e in zip(axs.ravel(), random.Random(0).sample(rows, 6)):
    ax.imshow(Image.open(f"data/{e['image_path']}"))
    for o in e["objects"]:
        b = json.loads(objects_to_answer([o]))[0]["bbox_2d"]
        x1, y1, x2, y2 = box_to_pixels(b)
        ax.add_patch(pa.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="r", lw=2))
        ax.text(x1, y1, o["label"], color="yellow", fontsize=8)
    ax.axis("off")
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000
Hàm `yolo_to_box`, `objects_to_answer`, `make_prompt` nằm trong `lab_lib.py`.

In [ ]:
e = rows[0]
print(make_prompt())
print(objects_to_answer(e["objects"]))

## 3. E0: zero-shot

In [ ]:
model, processor = load_model()
run_inference(model, processor, "val", "work/e0_val.jsonl", limit=40)   # gỡ lỗi trên 40 ảnh val
!python scripts/score.py --pred work/e0_val.jsonl --split val --name E0val --out work/e0_val.json
print(open("work/e0_val.jsonl").read()[:1500])   # xem output thô

In [ ]:
run_inference(model, processor, "test", "work/e0.jsonl")
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json
import gc, torch
del model; gc.collect(); torch.cuda.empty_cache()

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
model, processor = load_model()
ex = TrainSet(processor)[0]
img_id = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
print("token ảnh:", (ex["input_ids"] == img_id).sum().item(), "(phải = 220)")
lab = ex["labels"][0]
print(repr(processor.tokenizer.decode(lab[lab != -100])))   # phải là JSON đáp án + <|im_end|>
del model; gc.collect(); torch.cuda.empty_cache()

## 5. Setup LoRA / đóng băng cho E1, E2, E3
`apply_setup` nằm trong `lab_lib.py`, được gọi bên trong `train()`. Kiểm tra: E1/E2 = 252 module, E3 = 348.

## 6. Vòng huấn luyện

In [ ]:
runs = {}
runs["E1"] = train("E1", "ckpt/e1")

In [ ]:
runs["E2"] = train("E2", "ckpt/e2")

In [ ]:
runs["E3"] = train("E3", "ckpt/e3")

## 7. Suy luận và chấm E1–E3

In [ ]:
def eval_setup(name, ckpt, split="test"):
    pm, proc = load_trained(ckpt)
    out = f"work/{name.lower()}.jsonl"
    run_inference(pm, proc, split, out)
    r = json.load(open(f"{ckpt}/run.json"))
    !python scripts/score.py --pred {out} --name {name} --out report/evidence/{name.lower()}.json \
        --meta trainable_params={r['trainable_params']} --meta peak_vram_gb={r['peak_vram_gb']} --meta train_minutes={r['train_minutes']}
    del pm; gc.collect(); torch.cuda.empty_cache()

eval_setup("E1", "ckpt/e1")

In [ ]:
eval_setup("E2", "ckpt/e2")

In [ ]:
eval_setup("E3", "ckpt/e3")

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)
Ví dụ ablation rank trên setup tốt nhất (đổi `E3` nếu E1/E2 tốt hơn): chạy r=4 và r=64, `alpha = 2*r`. Lưu kết quả thành `report/evidence/e4_rank_<giá trị>.json`.

In [ ]:
r4 = train("E3", "ckpt/e4_r4", r=4, alpha=8)

In [ ]:
pm, proc = load_trained("ckpt/e4_r4")
run_inference(pm, proc, "test", "work/e4_r4.jsonl")
!python scripts/score.py --pred work/e4_r4.jsonl --name E4_rank_4 --out report/evidence/e4_rank_4.json \
    --meta trainable_params={r4['trainable_params']} --meta peak_vram_gb={r4['peak_vram_gb']} --meta train_minutes={r4['train_minutes']}